# Lesson 07 - 계획 디자인 패턴

이 노트북은 Microsoft Agent Framework를 사용한 AI 에이전트를 위한 **계획 디자인 패턴**을 보여줍니다.  
복잡한 여행 요청을 구조화된 하위 작업으로 분해하고, 이를 전문 에이전트에 할당하며,  
Pydantic 모델로 구동되는 구조화된 출력을 사용하여 결과 계획을 실행하는 방법을 배우게 됩니다.


## 설정


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 작업 분해

작업 분해는 계획 설계 패턴의 핵심입니다. 단일 에이전트에게 복잡한 요청을 처음부터 끝까지 처리하도록 요청하는 대신, 문제를 더 작고 명확하게 정의된 **하위 작업**으로 분할합니다. 각 하위 작업은 명확한 우선순위와 의존성 순서가 있는 전문 에이전트(예: 항공편, 호텔, 활동)에게 할당됩니다.

이 접근법은 다음과 같은 여러 이점을 제공합니다:
- **명확성**: 각 하위 작업은 단일 책임을 가집니다.
- **병렬성**: 독립적인 하위 작업은 동시에 실행될 수 있습니다.
- **신뢰성**: 실패가 개별 하위 작업에 국한됩니다.
- **예산 추적**: 비용이 하위 작업별로 추정되어 통합됩니다.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## 구조화된 출력을 사용하는 계획 에이전트 만들기

계획 에이전트는 **프런트 데스크 코디네이터** 역할을 합니다. 상위 수준의 여행 요청이 주어지면 구조화된 `TravelPlan`을 생성하여 요청을 하위 작업으로 분해하고, 우선순위를 설정하며, 컨시어지나 실행 계층이 작업을 수행할 수 있도록 종속성을 식별합니다.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## 전문가 도구를 사용한 계획 실행

프런트 데스크 담당자가 구조화된 계획을 수립하면, **컨시어지 에이전트**가 이를 실행합니다.
각 전문 도구는 한 가지 카테고리의 하위 작업(항공편, 호텔, 활동)을 처리합니다. 컨시어지는 계획의 하위 작업을 의존성 순서대로 반복하여 각 작업을 적절한 도구에 할당합니다.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## 요약

이번 강의에서는 AI 에이전트를 위한 **계획 설계 패턴**에 대해 배웠습니다:

1. **작업 분해** — 프론트 데스크 계획 에이전트가 복잡한 여행 요청을 Pydantic 모델을 사용해 구조화된 하위 작업으로 나누고, 각 작업을 우선순위와 종속성을 고려하여 전문가 에이전트에게 할당합니다.
2. **구조화된 출력** — `response_format`을 전달하여 에이전트가 자유 형식 텍스트 대신 검증된 `TravelPlan` 객체를 반환하도록 하여 후속 처리의 신뢰성을 높입니다.
3. **계획 실행** — 컨시어지 에이전트가 전문가 도구(`book_flight`, `reserve_hotel`, `book_activity`)를 사용해 하위 작업을 순차적으로 수행하고 결과를 보고합니다.

이 패턴은 *무엇을 할지* (계획)와 *어떻게 할지* (실행)를 분리하여 에이전트를 더 모듈화하고, 테스트하기 쉽고, 확장하기 용이하게 만듭니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 최선을 다하고 있으나, 자동 번역은 오류나 부정확성이 있을 수 있음을 유념해 주시기 바랍니다. 원문은 해당 언어의 원본 문서가 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문적인 인간 번역을 권장합니다. 이 번역본의 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
